# Training and Evaluation in one Notebook for One Model-Database Pair

# To check before running
1. Check class names for your event log in the **p2pencoder.py** ( *{event_log_name}encoder.py* )
2. Check the Axioms in **axiombuilder.py**
3. make sure you have done the declare mining on the event log and have a valid **ltn_rows_path**

In [41]:
event_log_name = "wide"
if event_log_name is None:
    raise ValueError("Please set the event_log_name variable to the name of the event log you want to use.")
ltn_rows_path = f"{event_log_name}_ltn_rows.pkl"
print(f"Event log name {event_log_name}")
print(f"LTN Rows path {ltn_rows_path}")
# starting time


Event log name wide
LTN Rows path wide_ltn_rows.pkl


In [42]:
# import tensorflow as tf
# physical_devices = tf.config.list_physical_devices('GPU')
# print(physical_devices)
# if len(physical_devices) > 0:
#     tf.config.experimental.set_memory_growth(physical_devices[0], True)
#     print("GPU found")
#     print("Memory growth set")
# else:
#     print("No GPU found")

In [43]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

import itertools

from sklearn import metrics


from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.wideevaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

import matplotlib.pyplot as plt
import numpy as np
np.random.seed(0)

import pandas as pd
import seaborn as sns
from sqlalchemy.orm import Session
import scikit_posthocs as sp

from april.database import get_engine
from april.fs import PLOT_DIR
from april.utils import microsoft_colors, prettify_dataframe, cd_plot, get_cd
from april.enums import Base, Strategy, Heuristic

sns.set_style('white')
pd.set_option('display.max_rows', 50)
%config InlineBackend.figure_format = 'retina'
print(wide_leaky_row_classes)

[<class 'april.anomalydetection.wideencoder.WideDAE-Leaky-10'>, <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-25'>, <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-50'>, <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-100'>, <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-150'>, <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-200'>, <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-250'>, <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-300'>]


In [44]:
dataset = f"{event_log_name}-0.3-1"
out_dir = PLOT_DIR / f'{event_log_name}_evaluations_both_{arrow.now().format("YYYY-MM-DD-HH-mm-ss")}'
eval_file = out_dir / f'{event_log_name}_fraction_evaluations.pkl'
csv_file = out_dir / f'{event_log_name}_fraction_evaluations.csv'
excel_file = out_dir / f'{event_log_name}_fraction_evaluations.xlsx'
model_folder = r"D:\LTNcoder\.out\models"
db = r"D:\LTNcoder\.out\april.db"

# create out_dir if it does not exist
if not out_dir.exists():
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {out_dir}")
from april.utils import delete_all_files_in_folder, delete_evaluation_and_model_tables
delete_all_files_in_folder(model_folder)
delete_evaluation_and_model_tables(db)
start_time = arrow.now("Europe/Berlin")


Created directory: d:\LTNcoder\.out\plots\wide_evaluations_both_2025-08-17-18-38-10
Deleted all rows from Evaluation and Model tables.


# Training

In [45]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    dataset = Dataset(dataset_name)

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()
    pass

In [46]:
ads = [
        dict(ad=WideDAE, fit_kwargs=dict(epochs=6, batch_size=100)),
    ] + \
    [
        dict(ad=LEAKY_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100)) for LEAKY_ROW_CLASS 
        in wide_leaky_row_classes
    ] + \
    [
        dict(ad=LTN_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100, epochs_ltn=3))
        for LTN_ROW_CLASS in wide_ltn_row_classes
    ]
print(ads)
for ad in tqdm(ads, desc="Fitting ADs"):
    fit_and_save(dataset, **ad)


[{'ad': <class 'april.anomalydetection.wideencoder.WideDAE'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-10'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-25'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-50'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-100'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-150'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-200'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.wideencoder.WideDAE-Leaky-250'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.wideencoder.Wi

Fitting ADs:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 1/6
40/40 [==============================] - 1s 14ms/step - loss: 0.2234 - accuracy: 0.0433 - val_loss: 0.1588 - val_accuracy: 0.0000e+00
Epoch 2/6
40/40 [==============================] - 0s 6ms/step - loss: 0.0455 - accuracy: 0.1625 - val_loss: 0.0048 - val_accuracy: 0.0000e+00
Epoch 3/6
40/40 [==============================] - 0s 6ms/step - loss: 0.0047 - accuracy: 0.2453 - val_loss: 0.0044 - val_accuracy: 0.0000e+00
Epoch 4/6
40/40 [==============================] - 0s 6ms/step - loss: 0.0045 - accuracy: 0.2756 - val_loss: 0.0044 - val_accuracy: 0.0000e+00
Epoch 5/6
40/40 [==============================] - 0s 6ms/step - loss: 0.0044 - accuracy: 0.2858 - val_loss: 0.0043 - val_accuracy: 0.0000e+00
Epoch 6/6
40/40 [==============================] - 0s 7ms/step - loss: 0.0044 - accuracy: 0.3047 - val_loss: 0.0043 - val_accuracy: 0.0000e+00
d:\LTNcoder\.out\models\wide-0.3-1_widedae_20250817-183810.336025.keras
Loading model wide-0.3-1_widedae_20250817-183810.336025 / <april.fs.M

In [47]:
print(AD) #Evaluator dependso on AD

{'binetv0': <class 'april.anomalydetection.binet.binet.BINetv0'>, 'binetv1': <class 'april.anomalydetection.binet.binet.BINetv1'>, 'binetv2': <class 'april.anomalydetection.binet.binet.BINetv2'>, 'binetv3': <class 'april.anomalydetection.binet.binet.BINetv3'>, 'likelihood': <class 'april.anomalydetection.boehmer.BoehmerLikelihoodAnomalyDetector'>, 'dae': <class 'april.anomalydetection.autoencoder.DAE'>, 'daeltn': <class 'april.anomalydetection.autoencoder.DAELTN'>, 'daeltnfrozen': <class 'april.anomalydetection.autoencoder.DAELTNFROZEN'>, 'likelihood+': <class 'april.anomalydetection.boehmer.LikelihoodPlusAnomalyDetector'>, 'naive': <class 'april.anomalydetection.bezerra.NaiveAnomalyDetector'>, 'naive+': <class 'april.anomalydetection.bezerra.NaivePlusAnomalyDetector'>, 'one-class-svm': <class 'april.anomalydetection.basic.OneClassSVM'>, 'perfect': <class 'april.anomalydetection.basic.PerfectAnomalyDetector'>, 'random': <class 'april.anomalydetection.basic.RandomAnomalyDetector'>, 'sam

# Evaluation

In [48]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [49]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    print(f"{e} loaded.")
    # print attributes of e
    print(f"e.model_file: {e.model_file}")
    print(f"e.model_name: {e.model_name}")
    print(f"e.eventlog_name: {e.eventlog_name}")
    print(f"e.dataset: {e.dataset}")
    print(f"e.result: {e.result}")
    
    
    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        # print(f"Adding parameters: {e}, {base}, {heuristic}, {strategy}")
        _params.append([e, base, heuristic, strategy])
    
    print(f"{_params} parameters to evaluate.")

    return [_e for p in _params for _e in _evaluate(p)]

In [50]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(f"Available Models: {models}")
evaluations = []
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

Available Models: ['wide-0.3-1_widedae-leaky-100_20250817-183821.443245', 'wide-0.3-1_widedae-leaky-10_20250817-183813.302734', 'wide-0.3-1_widedae-leaky-150_20250817-183825.117052', 'wide-0.3-1_widedae-leaky-200_20250817-183828.107237', 'wide-0.3-1_widedae-leaky-250_20250817-183831.013457', 'wide-0.3-1_widedae-leaky-25_20250817-183815.986790', 'wide-0.3-1_widedae-leaky-300_20250817-183833.923650', 'wide-0.3-1_widedae-leaky-50_20250817-183818.730522', 'wide-0.3-1_widedae_20250817-183810.336025', 'wide-0.3-1_wideltnfrozen-100_20250817-183923.660710', 'wide-0.3-1_wideltnfrozen-10_20250817-183836.932309', 'wide-0.3-1_wideltnfrozen-150_20250817-183939.091597', 'wide-0.3-1_wideltnfrozen-200_20250817-184003.839918', 'wide-0.3-1_wideltnfrozen-250_20250817-184029.383340', 'wide-0.3-1_wideltnfrozen-25_20250817-183852.210684', 'wide-0.3-1_wideltnfrozen-300_20250817-184050.250515', 'wide-0.3-1_wideltnfrozen-50_20250817-183907.950542']


Evaluate:   0%|          | 0/17 [00:00<?, ?it/s]

Evaluating wide-0.3-1_widedae-leaky-100_20250817-183821.443245...
Loading model wide-0.3-1_widedae-leaky-100_20250817-183821.443245 / <april.fs.ModelFile object at 0x0000028772D3D520> for event log wide-0.3-1 at path d:\LTNcoder\.out\models\wide-0.3-1_widedae-leaky-100_20250817-183821.443245.keras
Self.ad_: <april.anomalydetection.wideencoder.WideDAE-Leaky-100 object at 0x0000028772D3D8B0>
<april.wideevaluator.Evaluator object at 0x0000028772D3D850> loaded.
e.model_file: d:\LTNcoder\.out\models\wide-0.3-1_widedae-leaky-100_20250817-183821.443245.keras
e.model_name: wide-0.3-1_widedae-leaky-100_20250817-183821.443245
e.eventlog_name: wide-0.3-1
Filtering dataset to 610 LTN rows.
Indices: [21, 38, 39, 60, 64, 95, 108, 117, 119, 134, 141, 142, 143, 149, 155, 156, 174, 185, 215, 237, 244, 255, 257, 262, 267, 268, 277, 295, 299, 307, 323, 324, 336, 359, 361, 364, 370, 377, 380, 389, 390, 393, 404, 417, 421, 425, 434, 445, 451, 460, 469, 475, 477, 480, 488, 492, 498, 499, 502, 510, 526, 545,

In [52]:

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

  0%|          | 0/4896 [00:00<?, ?it/s]

In [53]:
synth_datasets = ['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide']
bpic_datasets = ['bpic12', 'bpic13', 'bpic15', 'bpic17']
anonymous_datasets = ['real']
datasets = synth_datasets + bpic_datasets + anonymous_datasets
dataset_types = ['Synthetic', 'Real-life']

orig_ads = [ad['ad'].__name__ for ad in ads if "DAE" in ad['ad'].__name__]
new_ads = [ad['ad'].__name__ for ad in ads if "DAE" not in ad['ad'].__name__]
ads = orig_ads + new_ads

heuristics = [r'$best$', r'$default$', r'$elbow_\downarrow$', r'$elbow_\uparrow$', 
              r'$lp_\leftarrow$', r'$lp_\leftrightarrow$', r'$lp_\rightarrow$']
print(ads)

['WideDAE', 'WideDAE-Leaky-10', 'WideDAE-Leaky-25', 'WideDAE-Leaky-50', 'WideDAE-Leaky-100', 'WideDAE-Leaky-150', 'WideDAE-Leaky-200', 'WideDAE-Leaky-250', 'WideDAE-Leaky-300', 'WideLTNFROZEN-10', 'WideLTNFROZEN-25', 'WideLTNFROZEN-50', 'WideLTNFROZEN-100', 'WideLTNFROZEN-150', 'WideLTNFROZEN-200', 'WideLTNFROZEN-250', 'WideLTNFROZEN-300']


In [54]:
evaluation = evaluation.query(f'ad in {ads} and label == "Anomaly"')

In [55]:
display(evaluation)

,file_name,date,hyperparameters,training_duration,training_host,ad,dataset_name,process_model,noise,dataset_id,axis,base,heuristic,strategy,label,attribute_name,perspective,precision,recall,f1
1,wide-0.3-1_widedae-leaky-100_20250817-183821.4...,2025-08-17 18:38:24.671250,"{'epochs': 6, 'batch_size': 100}",3.228005,Dev-RTX,WideDAE-Leaky-100,wide-0.3-1,wide,0.3,1,0,scores,best,single,Anomaly,name,Control Flow,0.951389,0.878205,0.913333
3,wide-0.3-1_widedae-leaky-100_20250817-183821.4...,2025-08-17 18:38:24.671250,"{'epochs': 6, 'batch_size': 100}",3.228005,Dev-RTX,WideDAE-Leaky-100,wide-0.3-1,wide,0.3,1,0,scores,best,single,Anomaly,user,Data,0.000000,0.000000,0.000000
5,wide-0.3-1_widedae-leaky-100_20250817-183821.4...,2025-08-17 18:38:24.671250,"{'epochs': 6, 'batch_size': 100}",3.228005,Dev-RTX,WideDAE-Leaky-100,wide-0.3-1,wide,0.3,1,1,scores,best,single,Anomaly,name,Control Flow,0.557895,0.507987,0.531773
7,wide-0.3-1_widedae-leaky-100_20250817-183821.4...,2025-08-17 18:38:24.671250,"{'epochs': 6, 'batch_size': 100}",3.228005,Dev-RTX,WideDAE-Leaky-100,wide-0.3-1,wide,0.3,1,1,scores,best,single,Anomaly,user,Data,0.000000,0.000000,0.000000
9,wide-0.3-1_widedae-leaky-100_20250817-183821.4...,2025-08-17 18:38:24.671250,"{'epochs': 6, 'batch_size': 100}",3.228005,Dev-RTX,WideDAE-Leaky-100,wide-0.3-1,wide,0.3,1,2,scores,best,single,Anomaly,name,Control Flow,0.557895,0.507987,0.531773
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4887,wide-0.3-1_wideltnfrozen-50_20250817-183907.95...,2025-08-17 18:39:23.315390,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",15.364848,Dev-RTX,WideLTNFROZEN-50,wide-0.3-1,wide,0.3,1,0,scores,stable_right,position_attribute,Anomaly,user,Data,0.250000,0.120690,0.162791
4889,wide-0.3-1_wideltnfrozen-50_20250817-183907.95...,2025-08-17 18:39:23.315390,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",15.364848,Dev-RTX,WideLTNFROZEN-50,wide-0.3-1,wide,0.3,1,1,scores,stable_right,position_attribute,Anomaly,name,Control Flow,0.677725,0.456869,0.545802
4891,wide-0.3-1_wideltnfrozen-50_20250817-183907.95...,2025-08-17 18:39:23.315390,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",15.364848,Dev-RTX,WideLTNFROZEN-50,wide-0.3-1,wide,0.3,1,1,scores,stable_right,position_attribute,Anomaly,user,Data,0.117647,0.040404,0.060150
4893,wide-0.3-1_wideltnfrozen-50_20250817-183907.95...,2025-08-17 18:39:23.315390,"{'epochs': 6, 'batch_size': 100, 'epochs_ltn': 3}",15.364848,Dev-RTX,WideLTNFROZEN-50,wide-0.3-1,wide,0.3,1,2,scores,stable_right,position_attribute,Anomaly,name,Control Flow,0.677725,0.456869,0.545802


In [56]:
evaluation['perspective-label'] = evaluation['perspective'] + '-' + evaluation['label']
evaluation['attribute_name-label'] = evaluation['attribute_name'] + '-' + evaluation['label']
evaluation['dataset_type'] = 'Synthetic'
evaluation.loc[evaluation['process_model'].str.contains('bpic'), 'dataset_type'] = 'Real-life'
evaluation.loc[evaluation['process_model'].str.contains('real'), 'dataset_type'] = 'Real-life'

In [57]:
_filtered_evaluation = evaluation.query(f'ad in {ads} and (strategy == "{Strategy.ATTRIBUTE}"'
                                       f' or (strategy == "{Strategy.SINGLE}" and process_model == "bpic12")'
                                       f' or (strategy == "{Strategy.SINGLE}" and ad == "Naive+"))')

In [58]:
filtered_evaluation = _filtered_evaluation.query(f'heuristic == "{Heuristic.DEFAULT}"'
                                                 f' or (heuristic == "{Heuristic.LP_MEAN}" and ad in {orig_ads})'
                                                 f' or (heuristic == "{Heuristic.LP_LEFT}" and ad in {new_ads})'
                                                )

In [59]:
df = filtered_evaluation.query('axis == 0')
df = prettify_dataframe(df)
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name', 'perspective'])[['precision', 'recall', 'f1']].mean().reset_index()
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name'])[['precision', 'recall', 'f1']].mean().reset_index()
df['f1'] = 2 * df['recall'] * df['precision'] / (df['recall'] + df['precision'])

df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model', 'dataset_name'], values=['precision', 'recall', 'f1'])
df = df.fillna(0)
df = df.stack(1).stack(1).reset_index()
df.to_excel(str(out_dir / 'table.xlsx'), index=False)

# drop rows in column "axis" which have value "Attribute"
df = df.query('axis != "Attribute"')

# df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model'], values=['precision', 'recall', 'f1'], aggfunc=np.mean)

df.to_excel(str(excel_file), index=False)
df.to_csv(str(csv_file), index=False)
print(df)

    axis                 ad process_model dataset_name        f1  precision  \
0   Case            WideDAE          Wide   wide-0.3-1  0.537135   0.644068   
1   Case   WideDAE-Leaky-10          Wide   wide-0.3-1  0.539032   0.642857   
2   Case  WideDAE-Leaky-100          Wide   wide-0.3-1  0.541268   0.649254   
3   Case  WideDAE-Leaky-150          Wide   wide-0.3-1  0.535881   0.650794   
4   Case  WideDAE-Leaky-200          Wide   wide-0.3-1  0.542857   0.653846   
5   Case   WideDAE-Leaky-25          Wide   wide-0.3-1  0.484155   0.625000   
6   Case  WideDAE-Leaky-250          Wide   wide-0.3-1  0.539759   0.644928   
7   Case  WideDAE-Leaky-300          Wide   wide-0.3-1  0.539759   0.644928   
8   Case   WideDAE-Leaky-50          Wide   wide-0.3-1  0.539032   0.642857   
9   Case   WideLTNFROZEN-10          Wide   wide-0.3-1  0.351941   0.598039   
10  Case  WideLTNFROZEN-100          Wide   wide-0.3-1  0.572110   0.596774   
11  Case  WideLTNFROZEN-150          Wide   wide-0.3

C:\Users\devas\AppData\Local\Temp\ipykernel_9160\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()
C:\Users\devas\AppData\Local\Temp\ipykernel_9160\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()


In [60]:
display(df)

,axis,ad,process_model,dataset_name,f1,precision,recall
0,Case,WideDAE,Wide,wide-0.3-1,0.537135,0.644068,0.460654
1,Case,WideDAE-Leaky-10,Wide,wide-0.3-1,0.539032,0.642857,0.464080
2,Case,WideDAE-Leaky-100,Wide,wide-0.3-1,0.541268,0.649254,0.464080
3,Case,WideDAE-Leaky-150,Wide,wide-0.3-1,0.535881,0.650794,0.455460
4,Case,WideDAE-Leaky-200,Wide,wide-0.3-1,0.542857,0.653846,0.464080
5,Case,WideDAE-Leaky-25,Wide,wide-0.3-1,0.484155,0.625000,0.395115
6,Case,WideDAE-Leaky-250,Wide,wide-0.3-1,0.539759,0.644928,0.464080
7,Case,WideDAE-Leaky-300,Wide,wide-0.3-1,0.539759,0.644928,0.464080
8,Case,WideDAE-Leaky-50,Wide,wide-0.3-1,0.539032,0.642857,0.464080
9,Case,WideLTNFROZEN-10,Wide,wide-0.3-1,0.351941,0.598039,0.249337


# End

In [61]:
end_time = arrow.now("Europe/Berlin")
print(f"Start time: {start_time}")
print(f"End time: {end_time}")
print(f"Duration: {end_time - start_time}")

Start time: 2025-08-17T18:38:10.281067+02:00
End time: 2025-08-17T18:42:54.479783+02:00
Duration: 0:04:44.198716
